# 04 · One-stage anchor-based detector: RetinaNet

RetinaNet with a ResNet-50 FPN v2 backbone (levels P3–P7), COCO-pretrained (torchvision). There is no proposal stage: every anchor is classified directly, and the focal loss down-weights the huge number of easy background anchors. Experiments: anchor sizes derived from the object sizes in `01_eda`, then looser IoU thresholds for assigning anchors to small objects.

In [ ]:
import os, urllib.request
# helper modules shared by all notebooks (data split, metric, submission format, torchvision training)
REPO = "https://raw.githubusercontent.com/MakcDutch/AI-homework2/main/"
for f in ["common.py", "tv_train.py"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(REPO + f, f)

In [ ]:
import glob, time, numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
import common as C
import tv_train as TV

ROOT = C.find_data_root()
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("work")
os.makedirs(WORK, exist_ok=True)
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## Protocol
Same frames, masking, input size and metric as `02_yolo`: every 5th train frame, ignored regions painted grey,
long side resized to 1024, evaluation on all val frames with `common.coco_evaluate`. Optimiser for both torchvision
models: SGD (momentum 0.9, weight decay 1e-4), 500-iteration warm-up, cosine decay, mixed precision, horizontal flip.
A val subset (every 5th frame) is scored after each epoch for the learning curve.

In [ ]:
TRAIN_STEP = 5
IMGSZ = 1024
EPOCHS = 6
BATCH = 4
LR = 0.005
WORKERS = 4
DATA_DIR = "/tmp/yolo_data"

EXPERIMENTS = {
    "retina_base": dict(),                                         # anchors 32-512 px, IoU >= 0.5 positive, < 0.4 negative
    "retina_anchors": dict(anchor_sizes=(16, 32, 64, 128, 256)),   # halved: median object is ~25 px at 1024 input
    "retina_anchors_iou": dict(anchor_sizes=(16, 32, 64, 128, 256), fg_iou_thresh=0.4, bg_iou_thresh=0.3),
}
RUN = ["retina_base"]

In [ ]:
C.build_yolo_dataset(ROOT, DATA_DIR, train_step=TRAIN_STEP, val_step=5)
train_paths = sorted(glob.glob(f"{DATA_DIR}/images/train/*.jpg"))
val_frames = C.list_frames(ROOT, "val")
val_paths = val_frames["image"].tolist()
val_subset = val_frames[(val_frames["frame"] - val_frames.groupby("clip")["frame"].transform("min")) % 5 == 0]["image"].tolist()
test_paths = C.list_frames(ROOT, "test")["image"].tolist()
print(len(train_paths), "train |", len(val_paths), "val (", len(val_subset), "for curves ) |", len(test_paths), "test")

## Training, evaluation and submission

In [ ]:
def run_experiment(name, cfg):
    model = TV.build_retinanet(IMGSZ, **cfg)
    t0 = time.time()
    history = TV.train_model(model, train_paths, IMGSZ, EPOCHS, BATCH, LR, f"{WORK}/{name}",
                             val_paths=val_subset, root=ROOT, workers=WORKERS)
    train_hours = (time.time() - t0) / 3600
    pd.DataFrame(history).to_csv(f"{WORK}/history_{name}.csv", index=False)

    val_preds, ms = TV.predict(model, val_paths, IMGSZ, workers=WORKERS)
    metrics, per_class = C.coco_evaluate(val_preds, ROOT, "val")
    test_preds, _ = TV.predict(model, test_paths, IMGSZ, workers=WORKERS)

    C.save_preds(val_preds, f"{WORK}/val_preds_{name}.pkl")
    C.save_preds(test_preds, f"{WORK}/test_preds_{name}.pkl")
    C.make_submission(test_preds, ROOT, f"{WORK}/sub_{name}.csv")
    pd.Series(per_class, name=name).to_csv(f"{WORK}/per_class_{name}.csv")
    row = dict(exp=name, family="one-stage anchor-based", imgsz=IMGSZ, epochs=EPOCHS,
               train_hours=round(train_hours, 2), ms_per_img=round(ms, 1), **metrics)
    C.results_table(f"{WORK}/results.csv", row)
    del model; torch.cuda.empty_cache()
    return row


for name in RUN:
    print(run_experiment(name, EXPERIMENTS[name]))

## Results

In [ ]:
display(C.results_table(f"{WORK}/results.csv"))
per_class = pd.concat([pd.read_csv(f"{WORK}/per_class_{n}.csv", index_col=0) for n in RUN], axis=1)
display(per_class.round(3))

fig, ax = plt.subplots(figsize=(7, 3.5))
for name in RUN:
    h = pd.read_csv(f"{WORK}/history_{name}.csv")
    ax.plot(h["epoch"], h["val_mAP"], marker="o", label=name)
ax.set_xlabel("epoch"); ax.set_ylabel("val-subset mAP"); ax.legend()
plt.tight_layout(); plt.savefig(f"{WORK}/curves.png", dpi=120); plt.show()

## Do the anchors fit the objects?
For every train box (rescaled to 1024 px), the best IoU with any anchor shape when centres are aligned (as in YOLOv2).
A box whose best IoU is below the positive threshold gets no positive anchor and is never learned.

In [ ]:
_, anns = C.load_coco_frames(ROOT, "train")
anns = anns[anns["iscrowd"] == 0]
s = IMGSZ / np.maximum(anns["width"], anns["height"])
wh = np.c_[anns["w"] * s, anns["h"] * s]

def shapes(base_sizes):
    gen = TV.anchor_generator(base_sizes)
    return np.array([(sz / np.sqrt(r), sz * np.sqrt(r)) for level, ratios in zip(gen.sizes, gen.aspect_ratios)
                     for sz in level for r in ratios])  # (w, h); torchvision ratio = h / w

def best_iou(wh, anchors):
    inter = np.minimum(wh[:, None, 0], anchors[None, :, 0]) * np.minimum(wh[:, None, 1], anchors[None, :, 1])
    return (inter / (wh.prod(1)[:, None] + anchors.prod(1)[None] - inter)).max(1)

rows = []
for label, sizes in [("default 32-512", (32, 64, 128, 256, 512)), ("data-driven 16-256", (16, 32, 64, 128, 256))]:
    iou = best_iou(wh, shapes(sizes))
    small = np.sqrt(wh.prod(1)) < 16
    rows.append({"anchors": label, "boxes with IoU>=0.5, %": (iou >= 0.5).mean() * 100,
                 "boxes with IoU>=0.4, %": (iou >= 0.4).mean() * 100,
                 "boxes <16px with IoU>=0.5, %": (iou[small] >= 0.5).mean() * 100})
display(pd.DataFrame(rows).round(1))

## Failure cases
Ground truth (class colours), ignored regions (red hatch), predictions with score ≥ 0.3 (white dashed).

In [ ]:
preds = C.load_preds(f"{WORK}/val_preds_{RUN[-1]}.pkl")
C.show_failures(preds, ROOT, f"{WORK}/failures_{RUN[-1]}.png")